# Vanishing & exploding gradients

_Deep Learning (CS230)_

**Over long sequences, gradients shrink to nothing or blow up; clipping caps the blow-ups.**

Backprop through a long sequence multiplies many slopes together.

     If those slopes are small, the product shrinks toward 0: a vanishing gradient. The network can't learn long-range links.

---

This notebook is a practice scaffold for the **Vanishing & exploding gradients** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

# 20 sigmoid layers: gradients shrink as they flow back
layers = []
for _ in range(20):
    layers += [nn.Linear(10, 10), nn.Sigmoid()]
net = nn.Sequential(*layers)

x = torch.randn(4, 10)
loss = net(x).sum()
loss.backward()

first = net[0].weight.grad.abs().mean().item()    # gradient near the input
last = net[-2].weight.grad.abs().mean().item()     # gradient near the output
print("grad at last  layer:", round(last, 8))
print("grad at first layer:", round(first, 8))     # far smaller -> vanished

nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)   # cap explosions
print("clipped to a safe norm")

## Visualize the data & results

_In a real deep sigmoid network on digits, how small is the weight update by the time it reaches the first layer?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X, y = digits.data / 16.0, digits.target

deep = MLPClassifier(hidden_layer_sizes=(32,) * 8, activation="logistic",
                     solver="sgd", max_iter=1, warm_start=True,
                     random_state=0, learning_rate_init=0.5)
deep.partial_fit(X, y, classes=np.unique(y))
before = [w.copy() for w in deep.coefs_]
deep.partial_fit(X, y)
updates = [np.abs(a - b).mean() for a, b in zip(deep.coefs_, before)]

labels = ["L9 (out)", "L8", "L7", "L6", "L5", "L4", "L3", "L2", "L1 (in)"]
values = updates[::-1]                    # output-first
colors = ["#7ee787", "#7ee787", "#4ea1ff", "#4ea1ff", "#ffb454",
          "#ffb454", "#ff7b72", "#ff7b72", "#ff7b72"]

plt.bar(labels, values, color=colors)
plt.title("Real mean weight-update per layer, 8-layer sigmoid digit net")
plt.yscale("log")
plt.show()